In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

In [ ]:
ruta = '/content/drive/MyDrive/Colab Notebooks/2019-Oct.csv'

ecommerce_df = pd.read_csv(ruta)

print("Dimensiones del conjunto de datos:")
print(ecommerce_df.shape)

display(ecommerce_df.head())

In [ ]:
print("INFORMACION GENERAL")
ecommerce_df.info()

print("\nESTADISTICAS DESCRIPTIVAS")
display(ecommerce_df.describe(include='all').T)

print("\nVALORES UNICOS")
print(ecommerce_df.nunique())

print("\nVALORES FALTANTES")
print(ecommerce_df.isnull().sum())

In [ ]:
num_cols = ecommerce_df.select_dtypes(
    include=np.number
).columns.tolist()

print("Variables numericas:")
print(num_cols)

ecommerce_df[num_cols].hist(
    figsize=(15, 8),
    bins=30
)

plt.tight_layout()
plt.show()

In [ ]:
cat_cols = ecommerce_df.select_dtypes(
    include='object'
).columns.tolist()

print("Variables categoricas:")
print(cat_cols)

for columna in cat_cols:

    print("\n", columna)

    display(
        ecommerce_df[columna]
        .value_counts()
        .head(10)
    )

In [ ]:
faltantes = ecommerce_df.isnull().sum()

porcentaje = (
    ecommerce_df.isnull().mean() * 100
)

resumen_faltantes = pd.DataFrame({
    'Valores faltantes': faltantes,
    'Porcentaje': porcentaje
})

display(resumen_faltantes)

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    ecommerce_df.corr(numeric_only=True),
    annot=True,
    cmap='coolwarm'
)

plt.title('Matriz de correlacion')
plt.show()

In [ ]:
# Convertir precio a formato numerico
ecommerce_df['price'] = pd.to_numeric(
    ecommerce_df['price'],
    errors='coerce'
)

# Eliminar precios faltantes
ecommerce_df.dropna(
    subset=['price'],
    inplace=True
)

# Eliminar precios iguales o menores a cero
ecommerce_df = ecommerce_df[
    ecommerce_df['price'] > 0
].copy()

# Eliminar registros duplicados
ecommerce_df.drop_duplicates(inplace=True)

print("Dimensiones despues de la limpieza:")
print(ecommerce_df.shape)

In [ ]:
# Variable objetivo
y = ecommerce_df['price']

# Variables independientes
columnas_X = [
    'category_code',
    'brand'
]

X = ecommerce_df[columnas_X].copy()

print("Variables independientes:")
print(X.head())

print("\nVariable objetivo:")
print(y.head())

print("\nDimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=42
)

print("Datos de entrenamiento:", Xtrain.shape)
print("Datos de prueba:", Xtest.shape)

In [ ]:
num_features = Xtrain.select_dtypes(
    include=np.number
).columns.tolist()

cat_features = Xtrain.select_dtypes(
    exclude=np.number
).columns.tolist()


num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])


cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


transformers = []

if num_features:
    transformers.append(
        ('num', num_pipeline, num_features)
    )

if cat_features:
    transformers.append(
        ('cat', cat_pipeline, cat_features)
    )


preprocessing = ColumnTransformer(
    transformers=transformers
)

print("Pipeline de procesamiento creado correctamente.")

In [ ]:
lr_model = make_pipeline(
    preprocessing,
    LinearRegression()
)

lr_model.fit(Xtrain, ytrain)

print("Modelo entrenado correctamente.")

In [ ]:
predictions = lr_model.predict(Xtest)

resultados = pd.DataFrame({
    'Precio real': ytest.values,
    'Precio predicho': predictions
})

display(resultados.head(10))

In [ ]:
rmse = np.sqrt(
    mean_squared_error(ytest, predictions)
)

r2 = r2_score(ytest, predictions)

print("RESULTADOS DEL MODELO")
print("--------------------------")
print("RMSE:", round(rmse, 2))
print("R2:", round(r2, 4))

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x=ytest,
    y=predictions
)

limite_min = min(ytest.min(), predictions.min())
limite_max = max(ytest.max(), predictions.max())

plt.plot(
    [limite_min, limite_max],
    [limite_min, limite_max],
    '--'
)

plt.xlabel('Precio real')
plt.ylabel('Precio predicho')

plt.title('Valores reales vs valores predichos')

plt.show()

In [ ]:
modelo = lr_model.named_steps['linearregression']

nombres = (
    lr_model.named_steps['columntransformer']
    .get_feature_names_out()
)

coeficientes = pd.DataFrame({
    'Caracteristica': nombres,
    'Coeficiente': modelo.coef_
})

print("Intercepcion:")
print(modelo.intercept_)

print("\nCoeficientes:")
display(
    coeficientes.sort_values(
        by='Coeficiente',
        ascending=False
    )
)